# 第2回 宿題（実装）：Mac作業・ファイル管理ルール

個人の作業ルールを5項目のFAQにまとめ、LoRA学習前後の形式遵守率と内容正解率を比較します。
元教材: `notebooks/lora_format_tuning.ipynb`。モデル・固定版・学習条件・採点方法は教材に合わせています。

**実行方法**: Colabの「ランタイムのタイプを変更」でT4 GPUを選び、「すべてのセルを実行」。既に学習したランタイムならセッションを再起動してから実行してください。
最後に結果一式のZIPをダウンロードします。結果を保存してからランタイムを接続解除してください。

- 題材: 成果物の保存先、保存先が不明な場合、一時ファイル、後片付け、メールの下書き。
- 根拠なし質問: FAQに定めのない頻度、期間、例外、暗号化などを尋ねます。
- train / valid / test = 14 / 7 / 21件。質問文は重複させず、学習前後は同じtestを使います。
- `needs_review` はFAQに回答の根拠があるかを表します。「ユーザーに確認する」というルールがFAQに書かれている場合、そのルールの説明は `false` です。
- 内容採点は教材と同じ必須語句照合です。言い換えによる不合格や、不要な断定の追加は出力原文でも確認します。
- AI支援によりデータ・ノートブックを作成し、本人のColab環境で実行します。


## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.14.1+cu130'}


## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。
- `num_train_epochs: 20` は、学習データ（14 件）を 20 周する設定です。1 周が 2 ステップなので合計 40 ステップで、T4 で学習が 4〜5 分に収まります。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。

**見るところ**: 設定の JSON が表示されます。

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/lec02_mac_rules",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/lec02_mac_rules",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 20,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 96,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/lec02_mac_rules",
  "drive_dir": "/content/drive/MyDrive/aie2026/lec02_mac_rules"
}


## 2. 自分の題材と評価データ

本人の作業ルールから5項目を要約しました。各項目の質問4通りをtrain 2 / valid 1 / test 1に分け、test専用の言い換えをさらに2問ずつ加えます。
根拠なしはtrain 4 / valid 2 / test 6問です。FAQにないバックアップ、暗号化、保存期間、命名規則などを使い、既存ルールから勝手に推測しないかを確かめます。


In [4]:
FAQ = {
    "save": "成果物は作業対象のフォルダかその付近に保存し、ユーザーの指定があれば優先します。",
    "unclear": "保存先が不明な場合は、ユーザーに確認します。",
    "temporary": "一時ファイルは /tmp などの一時領域に作成でき、作成のたびに確認する必要はありません。",
    "cleanup": "処理の終了と必要ファイルの保存を確認後、今回作成した不要な一時ファイルだけを削除します。",
    "email": "送信の明示的な指定がないメールは、下書きの作成・保存までとします。",
}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

GROUNDED = {
    "save": (["成果物はどこに保存しますか", "作業で作った資料の置き場所を教えてください", "完成ファイルの保存場所はどこですか", "成果物の保存先のルールは？"], ["作業対象", "指定"]),
    "unclear": (["保存先がわからないときはどうしますか", "保存場所が不明な場合の対応は？", "置き場所を決められないときはどうしますか", "保存先に迷ったら誰に聞きますか"], ["ユーザー", "確認"]),
    "temporary": (["一時ファイルを作る場所と確認の要否を教えてください", "作業用の一時ファイルはどこに作れますか。毎回確認が必要ですか", "一時ファイルの作成場所と許可のルールは？", "一時ファイルを作成できる場所はどこですか"], ["一時領域"]),
    "cleanup": (["一時ファイルはいつ、どれを削除しますか", "作業終了時の一時ファイルの片付け方は？", "一時ファイルを削除する条件と対象を教えてください", "後片付けで消す一時ファイルの範囲は？"], ["確認", "今回作成", "不要"]),
    "email": (["送信の指定がないメールはどこまで作成しますか", "メール作成だけを頼まれたときはどうしますか", "送信を指示されていないメールの扱いは？", "送信の依頼がない場合、メールはどの状態にしますか"], ["下書き"]),
}
UNGROUNDED = [
    "バックアップは何日おきに行いますか", "成果物の暗号化にはどの方式を使いますか", "成果物のファイル名には必ず日付を付けますか",
    "完成した成果物の保管期間は何年ですか", "メールの署名には電話番号を入れますか", "一時ファイルの容量上限は何GBですか",
    "バックアップはどのクラウドサービスに保存しますか", "メールの下書きは何日後に削除しますか",
]
TEST_EXTRA_GROUNDED = {
    "save": ["ユーザーが成果物の保存先を指定したら、どこを優先しますか", "出来上がった資料の保存場所はどう決めますか"],
    "unclear": ["保存するフォルダがはっきりしない場合、何をすればよいですか", "適切な保存先を判断できない場合はどう対応しますか"],
    "temporary": ["変換途中の一時ファイルはどの場所に作成できますか", "一時ファイルを置ける領域と、作成ごとの確認の要否を教えてください"],
    "cleanup": ["処理に使った一時ファイルを消す前に何を確認し、どれを消しますか", "作業後に削除する一時ファイルの対象とタイミングを教えてください"],
    "email": ["メールを書いてほしいと言われただけなら、どう仕上げますか", "送信を明示されていないメールは下書きにしますか"],
}
TEST_EXTRA_UNGROUNDED = [
    "成果物を暗号化せず保存してよい例外はありますか", "一時ファイルを置くディスクの空き容量は最低何GB必要ですか",
    "メールの下書きの件名にはどの接頭辞を付けますか", "成果物のバックアップを外付けSSDに保存してもよいですか",
]
UNGROUNDED_ANSWER = "この FAQ には記載がありません。ユーザーに確認が必要です。"

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

def split_templates(templates):
    # テンプレートの先頭 2 つを train、3 つ目を valid、4 つ目を test に固定（言い換えを跨がせない）
    return templates[:2], templates[2:3], templates[3:4]

train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]   # 根拠なしの質問は 4 問ごとに train 2 / valid 1 / test 1 に振る
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["確認"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["確認"]}

# train は 14 件（根拠あり 10、根拠なし 4）。同じ例を何回学習するかは、データを複製せず、セクション 4 の num_train_epochs で決める
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "質問が train / valid / test のどこかで重複しています。テンプレートを見直してください"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("学習例の 1 件 | 質問:", train[0]["question"], "| 望ましい出力:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

train=14 valid=7 test=21 gold=21
学習例の 1 件 | 質問: 成果物はどこに保存しますか | 望ましい出力: {"answer": "成果物は作業対象のフォルダかその付近に保存し、ユーザーの指定があれば優先します。", "needs_review": false}


## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の 21 問に答えさせて採点します。

プロンプト（`SYSTEM`）には「必ず JSON 1 行で、`answer` と `needs_review` を返す」と書いてあります。つまり **形式は指示済み** です。このモデルがその指示を守れるかどうかが、学習前の確認点です。

**採点のしかた**（セクション 5 でも同じ関数を使います）

- **形式**: JSON として読め、キーが `answer` と `needs_review` のちょうど 2 つで、型が文字列と真偽値であること。
- **内容**: `needs_review` が正解と一致し、かつ正解の語句（例: 「作業対象」と「指定」）が `answer` に含まれること。

内容の判定には `needs_review` の一致が必要です。そのため、キーが欠けている出力は、文面が正しくても内容は × になります。文面そのものは、表で自分の目で確かめてください。

`Qwen/Qwen3.5-0.8B` は画像も扱えるモデルですが、`AutoModelForCausalLM` で読み込むとテキスト部分だけが使われます。

**見るところ**

- 1 つ目のセル: `重みのメモリ(GB): 3.01` のように表示されます。モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。
- 2 つ目のセル: 1 行の要約（`format_rate`、`content_rate` など）と、21 問の出力の表。評価には 1 分前後かかります。学習前は、多くの出力で `needs_review` が欠けているはずです。

**出ても問題ない警告**

- `You are sending unauthenticated requests to the HF Hub`: Hugging Face にログインせずにモデルを取りに行っている、という案内です。公開モデルはログインしなくてもダウンロードできます。ログインが要るのは次の場合で、モデルの大きさでは決まりません。
  - 利用規約への同意が必要なモデル（gated model。Llama や Gemma など）や、非公開のリポジトリを使うとき。
  - 自分のモデルやデータを Hub にアップロードするとき。
  - 短時間に大量のリクエストを送り、回数の上限に当たるとき。上限はログインしていないと IP アドレスごとに 5 分あたり 3,000 回のファイル取得（2025 年 9 月時点の公表値。[Hub の rate limits](https://huggingface.co/docs/hub/rate-limits)）で、ログインすると上がります。
  
  このノートブックのモデルは公開されていて同意も不要です。読み込みで送るリクエストは多くても数十回なので、上限には届きません。もし上限に当たって `429 Too Many Requests` が出ても、ライブラリが待ち時間を読み取って自動で再試行します。ログインが必要になったときは、Colab の左側の鍵アイコン（シークレット）に Hugging Face のトークンを `HF_TOKEN` という名前で登録し、このノートブックからのアクセスを許可すると、自動で使われます。
- `... is falling back to its reference PyTorch implementation ...`: このモデルの一部の層（線形注意）に対する高速な実装が入っていないので、遅いが正しい実装で動くという案内です。4 行ほど出ます。

**ここで読んでおくこと**: 表を見て、「どのキーが欠けているか」「文面は FAQ と合っているか」を確認します。この表はセクション 5 で学習後の出力と並べて見ます。出力の原文は `/content/lec02_mac_rules/eval_before.json` にも保存されています（左のファイル一覧から開けます）。

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 形式はプロンプトで指示している。モデルがこれを守れるかが学習前の確認点
SYSTEM = "あなたはMac作業ルール FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。FAQ に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。"

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [6]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def check_format(text: str):
    '''JSON として読め、必須キーと型が合うか。失敗理由も返す'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''内容の合否: needs_review が正解と一致し、必須語句を answer に含む'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''test の 21 問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 形式が壊れていても JSON らしい部分が読めるなら、内容だけは別に採点する
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は FAQ と合っているかを読む
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 21, "format_rate": 0.047619047619047616, "content_rate": 0.0, "both_rate": 0.0, "seconds": 45.9}


,質問,正解の needs_review,学習前の出力,形式,形式の失敗理由,内容
0,成果物の保存先のルールは？,False,"{""answer"": ""作業対象のフォルダかその付近に保存し、ユーザーの指定があれば優先します。""}",×,missing_key,×
1,ユーザーが成果物の保存先を指定したら、どこを優先しますか,False,"{""answer"": ""ユーザーの指定""}",×,missing_key,×
2,出来上がった資料の保存場所はどう決めますか,False,"{""answer"": ""作業対象のフォルダかその付近に保存し、ユーザーの指定があれば優先します。""}",×,missing_key,×
3,保存先に迷ったら誰に聞きますか,False,"{""answer"": ""ユーザーに確認します""}",×,missing_key,×
4,保存するフォルダがはっきりしない場合、何をすればよいですか,False,"{""answer"": ""ユーザーに確認してください""}",×,missing_key,×
5,適切な保存先を判断できない場合はどう対応しますか,False,"{""answer"": ""確認が必要""}",×,missing_key,×
6,一時ファイルを作成できる場所はどこですか,False,"{""answer"": ""/tmp""}",×,missing_key,×
7,変換途中の一時ファイルはどの場所に作成できますか,False,"{""answer"": ""/tmp""}",×,missing_key,×
8,一時ファイルを置ける領域と、作成ごとの確認の要否を教えてください,False,"{""answer"": ""/tmp"", ""needs_review"": false}",○,,×
9,後片付けで消す一時ファイルの範囲は？,False,"{""answer"": ""/tmp""}",×,missing_key,×


## 4. LoRA 学習（T4 で 4〜5 分）

### LoRA と QLoRA の違い

- **LoRA**: 基盤モデルの重みは固定し、一部の層に追加した小さな行列（adapter）だけを学習します。
- **QLoRA**: 基盤モデルの重みを 4bit に量子化して保持し、その上で同じように adapter だけを学習します。adapter 自体は量子化しません。

QLoRA の効果は、基盤モデルの重みを保持するメモリが減ることです。16bit の約 4 分の 1 になるので、そのままでは GPU に載らない大きさのモデルでも学習できます（70 億パラメータなら 16bit で約 14GB、4bit で約 3.5GB）。

| 観点 | LoRA（量子化なし） | QLoRA（4bit） |
| --- | --- | --- |
| 重みのメモリ | 大きい | 小さい（16bit の約 1/4） |
| 基盤モデルの出力 | 元のまま | 量子化の誤差で変わることがある。影響の大きさはモデルによって違う |
| 学習の速さ | 基準 | 速くはならない。4bit の重みを計算のたびに戻すので、遅くなることが多い |
| 依存ライブラリ | peft | peft に加えて bitsandbytes（CUDA の GPU が前提） |

QLoRA を実装するときの注意:

- 読み込み時に `BitsAndBytesConfig(load_in_4bit=True, ...)` を渡し、LoRA を付ける前に `prepare_model_for_kbit_training(model)` を通します。
- 保持精度（4bit）と計算精度（`bnb_4bit_compute_dtype`。16bit など）は別の設定です。
- 学習前後の比較は、同じ量子化条件の基盤モデルで行います。学習前を量子化なし、学習後を 4bit で測ると、量子化の影響と学習の影響が混ざります。
- adapter は学習時の基盤モデルに合わせて作られます。配布や再利用のときは、基盤モデルの ID・revision と量子化設定を一緒に記録します。
- 4bit での学習が勧められないモデルもあります。Qwen3.5 は量子化の誤差が通常より大きいとして、4bit 学習を勧めていない[学習ライブラリのガイド](https://unsloth.ai/docs/models/qwen3.5/fine-tune)があります。

この演習のモデルは量子化なしでも T4 に載るので、LoRA で学習します。モデルが GPU に載るなら LoRA、載らないなら QLoRA かより小さいモデル、という順で考えます。今回は実装課題に必要な量子化なしのLoRAだけを実行します。

### 学習の実行

**やること**: train の 14 件を chat 形式（system / user / assistant）に直し、LoRA を付けて 20 エポック（40 ステップ）学習します。学習した adapter は `adapter/` に保存します。

- `target_modules` は LoRA を差し込む層です。注意機構の射影（`q_proj` など）と MLP の射影（`gate_proj` など）を指定しています。このモデルは 24 層のうち 18 層が線形注意で、`q_proj` などを持つのは残りの 6 層だけです。MLP は全層にあります。
- バッチ 4 × 勾配累積 2 = 1 ステップ 8 件。train は 14 件なので 1 エポックは 2 ステップ（8 件と 6 件）で、20 エポックで 40 ステップです。各例をちょうど 20 回学習します。学習率の warmup と減衰（cosine）はこの 40 ステップに対して効きます。
- 学習後のモデルは変数 `tuned` に入ります。LoRA は基盤モデルの層に差し込まれるので、元の変数 `model` で生成しても adapter 込みの出力になります。学習前の出力が必要なときは、セクション 6 のように adapter を一時的に無効にします。

**見るところ**

- `trainable params: ... || all params: ... || trainable%: ...`: 学習するパラメータが全体の数 % だけであることが LoRA の特徴です。
- `Training Loss` の表（10 ステップごと）: 学習による損失の推移を確認します。値は題材と実行ごとに変わります。
- `学習時間: ... 秒 | ステップ数: 40 | エポック: 20.0`: T4 で 240〜290 秒です。ステップ数とエポックは、学習データ 14 件 × 20 周が上の計算どおりに実行されたことの確認です。

**出ても問題ない警告**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` は、終端トークンの設定を tokenizer に合わせたという案内です。

**うまくいかないとき**: 学習が途中で止まった、または学習できなかった場合は、下の 4b を読んでください。

In [7]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

token_lengths = [len(tok(tok.apply_chat_template(to_chat(e)["messages"], tokenize=False, enable_thinking=False), add_special_tokens=False)["input_ids"]) for e in train + valid]
print("学習・検証データの最大トークン数:", max(token_lengths))
assert max(token_lengths) <= CONFIG["max_seq_len"], "データがmax_seq_lenを超えています。FAQを短くしてから再実行してください"

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
TRAIN_SECONDS = round(time.time()-t0, 2)
print(f"学習時間: {TRAIN_SECONDS:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

Tokenizing train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

... (2 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,2.147618
20,0.166559
30,0.030581
40,0.017625


学習時間: 244 秒 | ステップ数: 40 | エポック: 20.0


### 4b. 学習が止まった・できなかった場合

同じランタイムにチェックポイントが残っている場合は、教材の手順に従って再開できます。ランタイムを削除した場合は、T4に再接続し、上から実行し直してください。
この宿題は独自のFAQを使うため、元の講座FAQの実行結果で代用しません。実行できなかった場合はその範囲を明記します。


## 5. 学習後評価（学習前と同じ test・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 21 問・同じ生成設定で評価し、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**

- 比較表: `format_rate`（形式遵守率）と `content_rate`（内容正解率）が学習前からどう変わったか。
- `退行`: 学習前は形式も内容も通っていたのに、学習後に落ちた質問です。空のリストが期待値です。
- 質問ごとの表: 残った失敗を読み、どの型かを見分けてください。
  - 根拠なしの質問（正解の `needs_review` が `True`）に、FAQ の別項目を当てはめて断定した。
  - 根拠ありの質問（正解が `False`）に、「記載がありません」と確認要求で逃げた。

学習後の回答は、教師データの文（FAQ の原文）をほぼそのまま返す形になります。形式と文面は揃いますが、「FAQ に根拠があるか」の判断まで学習できたとは限りません。

seed を固定しても、GPU 上の学習は完全には再現しません。形式遵守率と内容正解率の割合はほぼ同じでも、どの質問が残るかは実行ごとに変わることがあります。隣の人と「残った質問」が違っても異常ではないので、失敗の型で比べてください。

In [8]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 学習前は形式も内容も通ったのに、学習後に落ちた質問（退行）
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# 質問ごとの出力原文。形式（JSON・必須キー・型）と内容（needs_review の一致と必須語句）の判定を自分の目で確かめる
detail = pd.DataFrame([{"質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

{"label": "after", "n": 21, "format_rate": 1.0, "content_rate": 0.9523809523809523, "both_rate": 0.9523809523809523, "seconds": 38.2}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,21,0.047619,0.000000,0.000000,45.9
after,21,1.000000,0.952381,0.952381,38.2


退行: []


,質問,正解の needs_review,学習前の出力,学習前 形式/内容,学習後の出力,学習後 形式/内容
0,成果物の保存先のルールは？,False,"{""answer"": ""作業対象のフォルダかその付近に保存し、ユーザーの指定があれば優先します。""}",×/×,"{""answer"": ""成果物は作業対象のフォルダかその付近に保存し、ユーザーの指定があれば優先します。"", ""needs_review"": false}",○/○
1,ユーザーが成果物の保存先を指定したら、どこを優先しますか,False,"{""answer"": ""ユーザーの指定""}",×/×,"{""answer"": ""成果物は作業対象のフォルダかその付近に保存し、ユーザーの指定があれば優先します。"", ""needs_review"": false}",○/○
2,出来上がった資料の保存場所はどう決めますか,False,"{""answer"": ""作業対象のフォルダかその付近に保存し、ユーザーの指定があれば優先します。""}",×/×,"{""answer"": ""保存先が不明な場合は、ユーザーに確認します。"", ""needs_review"": false}",○/×
3,保存先に迷ったら誰に聞きますか,False,"{""answer"": ""ユーザーに確認します""}",×/×,"{""answer"": ""保存先が不明な場合は、ユーザーに確認します。"", ""needs_review"": false}",○/○
4,保存するフォルダがはっきりしない場合、何をすればよいですか,False,"{""answer"": ""ユーザーに確認してください""}",×/×,"{""answer"": ""保存先が不明な場合は、ユーザーに確認します。"", ""needs_review"": false}",○/○
5,適切な保存先を判断できない場合はどう対応しますか,False,"{""answer"": ""確認が必要""}",×/×,"{""answer"": ""保存先が不明な場合は、ユーザーに確認します。"", ""needs_review"": false}",○/○
6,一時ファイルを作成できる場所はどこですか,False,"{""answer"": ""/tmp""}",×/×,"{""answer"": ""一時ファイルは /tmp などの一時領域に作成でき、作成のたびに確認する必要はありません。"", ""needs_review"": false}",○/○
7,変換途中の一時ファイルはどの場所に作成できますか,False,"{""answer"": ""/tmp""}",×/×,"{""answer"": ""一時ファイルは /tmp などの一時領域に作成でき、作成のたびに確認する必要はありません。"", ""needs_review"": false}",○/○
8,一時ファイルを置ける領域と、作成ごとの確認の要否を教えてください,False,"{""answer"": ""/tmp"", ""needs_review"": false}",○/×,"{""answer"": ""一時ファイルは /tmp などの一時領域に作成でき、作成のたびに確認する必要はありません。"", ""needs_review"": false}",○/○
9,後片付けで消す一時ファイルの範囲は？,False,"{""answer"": ""/tmp""}",×/×,"{""answer"": ""処理の終了と必要ファイルの保存を確認後、今回作成した不要な一時ファイルだけを削除します。"", ""needs_review"": false}",○/○


## 6. 知識を問う質問（形式を学んでも知識は保証されない）

**やること**: 形式の実験とは別に、FAQ にも学習データにも無い事実を 3 問尋ねます（対照として FAQ にある質問を 1 問加えます）。望ましい応答は「確認が必要（`needs_review: true`）」です。

応答を 3 つに分類します。

| 分類 | 意味 |
| --- | --- |
| `format_error` | JSON の形式を守れていない（学習前に多い） |
| `needs_review` | 「記載がありません。確認が必要です」と返した（望ましい応答） |
| `asserted` | `needs_review: false` で言い切った。FAQ に無い質問に対してなら、根拠のない断定（創作）です |

**見るところ**: 学習前は形式違反になりやすく、学習後は「確認が必要」と「断定」の両方が出ることがあります。出力の原文も表に出すので、断定した場合に何を根拠にしたように見えるか（FAQ の別項目を流用しているか）を読んでください。「Fine-Tuning では知識は絶対に直らない」と先に結論にせず、今回のデータでの結果を報告します。

学習前の出力は、adapter を一時的に無効にして取っています（`tuned.disable_adapter()`）。

In [9]:
KNOWLEDGE_PROBES = [
    ("バックアップ先のストレージ容量は何GBですか", False),
    ("一時ファイルの自動削除は毎日何時に実行しますか", False),
    ("メールの承認担当者の氏名は何ですか", False),
    ("送信の指示がないメールはどこまで作ればよいですか", True),
]
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

knowledge_rows = []
for q, is_grounded in KNOWLEDGE_PROBES:
    ex = make_example(q, "", True)
    # LoRA は基盤モデルの層に差し込まれるので、学習後は `model` で生成しても adapter 込みになる。学習前の出力は adapter を一時的に無効にして取る
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"質問": q, "FAQ にある": is_grounded, "学習前の分類": classify(before_raw), "学習前の出力": before_raw,
                 "学習後の分類": classify(after_raw), "学習後の出力": after_raw})
display(pd.DataFrame(knowledge_rows))

,質問,FAQ にある,学習前の分類,学習前の出力,学習後の分類,学習後の出力
0,バックアップ先のストレージ容量は何GBですか,False,format_error,"{""answer"": ""確認が必要""}",needs_review,"{""answer"": ""この FAQ には記載がありません。ユーザーに確認が必要です。"", ""needs_review"": true}"
1,一時ファイルの自動削除は毎日何時に実行しますか,False,format_error,"{""answer"": ""確認が必要""}",needs_review,"{""answer"": ""この FAQ には記載がありません。ユーザーに確認が必要です。"", ""needs_review"": true}"
2,メールの承認担当者の氏名は何ですか,False,format_error,"{""answer"": ""確認が必要""}",needs_review,"{""answer"": ""この FAQ には記載がありません。ユーザーに確認が必要です。"", ""needs_review"": true}"
3,送信の指示がないメールはどこまで作ればよいですか,True,format_error,"{""answer"": ""確認が必要""}",asserted,"{""answer"": ""送信の明示的な指定がないメールは、下書きの作成・保存までとします。"", ""needs_review"": false}"


## 7. 結果の保存

実測した設定・学習ログ・学習前後21問の出力・追加4問の結果を `result.json` に保存します。
最後に `lec02_mac_rules_results.zip` がダウンロードされます。展開した結果をリポジトリの `lectures/02/results/mac_rules/` に置き、`answer.md` の「宿題」欄に比較と考察を記入します。


In [11]:
# トークン数は入力IDの長さで確認する（tokenizerの辞書のキー数では数えない）。
token_lengths = [len(tok(tok.apply_chat_template(to_chat(e)["messages"], tokenize=False, enable_thinking=False), add_special_tokens=False)["input_ids"]) for e in train + valid]
print("学習・検証データの最大トークン数:", max(token_lengths))
assert max(token_lengths) <= CONFIG["max_seq_len"], "入力がmax_seq_lenを超えています。条件を見直して再実行してください"
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "topic": "Mac作業・ファイル管理ルール",
    "execution_support": "AI支援で作成し、本人のColab環境で実行",
    "dataset_sizes": {"train": len(train), "valid": len(valid), "test": len(test)},
    "training_seconds": TRAIN_SECONDS,
    "training_steps": trainer.state.global_step,
    "training_log": trainer.state.log_history,
    "max_training_tokens": max(token_lengths),
    "faq": FAQ,
    "before_rows": before_rows, "after_rows": after_rows,
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k not in ("knowledge", "before_rows", "after_rows", "faq", "training_log")}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "train.jsonl", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])
from zipfile import ZipFile, ZIP_DEFLATED
bundle = WORK / "lec02_mac_rules_results.zip"
with ZipFile(bundle, "w", ZIP_DEFLATED) as z:
    for name in ["result.json", "eval_before.json", "eval_after.json", "train.jsonl", "test.jsonl", "gold.json"]:
        z.write(WORK / name, arcname=name)
print("完了: 結果ZIPをダウンロードします。保存後にランタイムを接続解除してください。")
from google.colab import files
files.download(str(bundle))


学習・検証データの最大トークン数: 250
{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 20,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.14.1+cu130"
  },
  "method": "LoRA（量子化なし）",
  "topic": "Mac作業・ファイル管理ルール",
  "execution_support": "AI支援で作成し、本人のColab環境で実行",
  "dataset_sizes": {
    "train": 14,
    "valid": 7,
    "test": 21
  },
  "training_seconds": 243.65,
  "training_steps": 40,
  "max_training_tokens": 250,
  "before": {
    "label": "before",
    "n": 21,
    "format_rate": 0.047619047619047616,
    "content_rate": 0.0,
    "both_rate": 0.0,
    "seconds": 45.9
  },
  "after": {
    "label": "after",
    "n": 21,
    "format_rate": 1.0,
    "content_rate"

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>